# Confusion Matrix Query Benchmark

Measures query latency for two scenarios:

- **(a) Many rows, 2 shards** — `shard_id` filter targets 2 physical shards; wide grid range returns many rows to merge.
- **(b) Few rows, many shards** — no `shard_id` filter (full fan-out); narrow grid window returns few rows per shard.

Because `shard_id` is the Citus distribution column, filtering on it enables shard pruning. Omitting it causes Citus to query every physical shard.

In [1]:
import time
import random
import statistics
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import psycopg
from psycopg.rows import dict_row
from constants import CITUS_HEAD_HOST, CITUS_HEAD_PORT, CITUS_HEAD_DB, CITUS_HEAD_USER, CITUS_HEAD_PASSWORD

ITERATIONS = 20   # repetitions per scenario
ROWS_PER_SHARD = 5_000   # synthetic rows inserted per shard for seeding

def get_conn():
    return psycopg.connect(
        host=CITUS_HEAD_HOST, port=CITUS_HEAD_PORT, dbname=CITUS_HEAD_DB,
        user=CITUS_HEAD_USER, password=CITUS_HEAD_PASSWORD,
        autocommit=True, row_factory=dict_row,
    )

## 1. Discover shards and seed synthetic data

Rows are inserted directly with synthetic `shard_id` values drawn from the live `confusion_matrix_ln` shard list so Citus routes them correctly.

In [2]:
with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT shardid
            FROM pg_dist_shard
            WHERE logicalrelid = 'confusion_matrix_ln'::regclass
            ORDER BY shardid;
        """)
        shard_ids = [r['shardid'] for r in cur.fetchall()]

print(f"Physical confusion_matrix_ln shards: {len(shard_ids)}")
print(f"Shard IDs: {shard_ids}")

Physical confusion_matrix_ln shards: 32
Shard IDs: [110796, 110797, 110798, 110799, 110800, 110801, 110802, 110803, 110804, 110805, 110806, 110807, 110808, 110809, 110810, 110811, 110812, 110813, 110814, 110815, 110816, 110817, 110818, 110819, 110820, 110821, 110822, 110823, 110824, 110825, 110826, 110827]


In [3]:
# Seed: truncate first, then bulk-insert synthetic rows spread across all shards.
# Grid values cover the full 0-4095 range so wide/narrow filters are meaningful.

LABEL_CLASSES = list(range(5))   # 5 synthetic label classes
GRID_MAX = 4095

with get_conn() as conn:
    with conn.cursor() as cur:
        cur.execute("TRUNCATE TABLE confusion_matrix_ln;")
        print("Truncated confusion_matrix_ln.")

        rows = []
        for sid in shard_ids:
            for _ in range(ROWS_PER_SHARD):
                rows.append((
                    sid,
                    random.randint(0, GRID_MAX),
                    random.randint(0, GRID_MAX),
                    random.choice(LABEL_CLASSES),
                    random.choice(LABEL_CLASSES),
                    random.randint(1, 100),
                ))

        cur.executemany("""
            INSERT INTO confusion_matrix_ln
                (shard_id, grid_cell_i, grid_cell_j, pred_label, gt_label, bucket_date, count)
            VALUES (%s, %s, %s, %s, %s, CURRENT_DATE, %s)
            ON CONFLICT (grid_cell_i, grid_cell_j, pred_label, gt_label, shard_id)
            DO UPDATE SET count = confusion_matrix_ln.count + EXCLUDED.count;
        """, rows)

        cur.execute("SELECT count(*) FROM confusion_matrix_ln;")
        total = cur.fetchone()['count']
        print(f"Seeded ~{len(rows)} rows ({total} after conflict merging) across {len(shard_ids)} shards.")

Truncated confusion_matrix_ln.
Seeded ~160000 rows (160000 after conflict merging) across 32 shards.


## 2. Define benchmark helper

In [4]:
def bench_query(label: str, sql: str, params: tuple, iterations: int = ITERATIONS):
    """Run `sql` with `params` for `iterations` repetitions and report latency stats."""
    latencies = []
    row_count = None
    with get_conn() as conn:
        with conn.cursor() as cur:
            for i in range(iterations):
                t0 = time.perf_counter()
                cur.execute(sql, params)
                rows = cur.fetchall()
                latencies.append(time.perf_counter() - t0)
                if row_count is None:
                    row_count = len(rows)

    mean_ms = statistics.mean(latencies) * 1000
    median_ms = statistics.median(latencies) * 1000
    stdev_ms = statistics.stdev(latencies) * 1000 if len(latencies) > 1 else 0
    p95_ms = sorted(latencies)[int(0.95 * iterations)] * 1000

    print(f"\n{'='*60}")
    print(f"Scenario: {label}")
    print(f"  Result rows   : {row_count}")
    print(f"  Mean          : {mean_ms:.2f} ms")
    print(f"  Median        : {median_ms:.2f} ms")
    print(f"  Stdev         : {stdev_ms:.2f} ms")
    print(f"  p95           : {p95_ms:.2f} ms")
    return latencies

## 3. Scenario (a) — Many rows over 2 shards

Filter on 2 `shard_id` values (Citus prunes to those 2 physical shards).  
Wide grid range captures most rows on those shards.

In [5]:
# Pick 2 shard IDs from opposite ends of the list for variety.
shard_a, shard_b = shard_ids[0], shard_ids[-1]
print(f"Targeting shards: {shard_a}, {shard_b}")

sql_a = """
    SELECT pred_label, gt_label, SUM(count) AS total
    FROM confusion_matrix_ln
    WHERE shard_id = ANY(%s)
      AND grid_cell_i BETWEEN %s AND %s
      AND grid_cell_j BETWEEN %s AND %s
    GROUP BY pred_label, gt_label
    ORDER BY pred_label, gt_label;
"""
# Wide grid window: full range
params_a = ([shard_a, shard_b], 0, GRID_MAX, 0, GRID_MAX)

latencies_a = bench_query("(a) many rows, 2 shards", sql_a, params_a)

Targeting shards: 110796, 110827

Scenario: (a) many rows, 2 shards
  Result rows   : 25
  Mean          : 10.91 ms
  Median        : 5.27 ms
  Stdev         : 25.04 ms
  p95           : 117.30 ms


## 4. Scenario (b) — Few rows over many shards

No `shard_id` filter — Citus fans out to all physical shards.  
Narrow grid window means each shard contributes very few rows.

In [9]:
# Narrow grid window: 1% of the total range on each axis
NARROW = GRID_MAX  # ~40 units
i_lo, i_hi = GRID_MAX // 2, GRID_MAX // 2 + NARROW
j_lo, j_hi = GRID_MAX // 2, GRID_MAX // 2 + NARROW
print(f"Grid window: i=[{i_lo},{i_hi}], j=[{j_lo},{j_hi}]")

sql_b = """
    SELECT pred_label, gt_label, SUM(count) AS total
    FROM confusion_matrix_ln
    WHERE grid_cell_i BETWEEN %s AND %s
      AND grid_cell_j BETWEEN %s AND %s
    GROUP BY pred_label, gt_label
    ORDER BY pred_label, gt_label;
"""
params_b = (i_lo, i_hi, j_lo, j_hi)

latencies_b = bench_query("(b) few rows, all shards", sql_b, params_b)

Grid window: i=[2047,6142], j=[2047,6142]

Scenario: (b) few rows, all shards
  Result rows   : 25
  Mean          : 20.64 ms
  Median        : 13.23 ms
  Stdev         : 31.94 ms
  p95           : 156.25 ms


## 5. Summary

In [7]:
import statistics

def summarise(name, lats):
    ms = [x * 1000 for x in lats]
    print(f"{name:40s}  mean={statistics.mean(ms):6.1f}ms  median={statistics.median(ms):6.1f}ms  p95={sorted(ms)[int(0.95*len(ms))]:6.1f}ms")

print(f"{'Scenario':40s}  {'mean':>10}  {'median':>10}  {'p95':>10}")
print("-" * 72)
summarise("(a) many rows, 2 shards", latencies_a)
summarise("(b) few rows, all shards", latencies_b)

Scenario                                        mean      median         p95
------------------------------------------------------------------------
(a) many rows, 2 shards                   mean=  10.9ms  median=   5.3ms  p95= 117.3ms
(b) few rows, all shards                  mean=   9.8ms  median=   3.1ms  p95= 134.9ms
